# Structural Tracking: ENFORCE Mode Violations

This notebook demonstrates how ENFORCE mode blocks structural violations.

When you:
1. Read a structural attribute (like `df.columns`)
2. Then modify the structure in the same cell

ENFORCE mode will detect this and raise a violation.

In [1]:
import pandas as pd

# Enable ENFORCE mode
%structural_tracking enforce

<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✅ Structural tracking mode set to: enforce</div>

<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Run: 6 ms | State: 4 ms | Check: 0 ms | Reads: open</div>

## Column Addition After Reading Columns

This pattern is problematic:
1. Read `df.columns`
2. Add a new column

The output of step 1 is now stale.

In [2]:
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Run: 2 ms | State: 2 ms | Check: 0 ms | Writes: df</div>

In [3]:
# This cell reads columns then adds one - violation!
print(f"Columns before: {list(df.columns)}")
df['c'] = [7, 8, 9]  # This changes the structure we just read
print(f"Columns after: {list(df.columns)}")

Columns before: ['a', 'b']
Columns after: ['a', 'b', 'c']


<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Run: 2 ms | State: 2 ms | Check: 1 ms | Reads: df | Writes: df[c]</div>

## Row Addition After Reading Shape

This pattern is also problematic:
1. Read `df.shape` or `len(df)`
2. Add rows

In [4]:
df2 = pd.DataFrame({'x': [1, 2]})

<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Run: 2 ms | State: 2 ms | Check: 0 ms | Writes: df2</div>

In [5]:
# This cell reads shape then adds rows - violation!
print(f"Shape before: {df2.shape}")
df2.loc[len(df2)] = [3]  # Adds a row
print(f"Shape after: {df2.shape}")

Shape before: (2, 1)
Shape after: (3, 1)


<div style='padding-left: 1em; font-size: 0.8em; background-color: #f0f0f8; margin-bottom: 0em;'>✓ Run: 2 ms | State: 2 ms | Check: 0 ms | Reads: df2</div>

## Length Violation

In [6]:
df = pd.DataFrame({'x': [1, 2, 3]})

SDCViolation: Cell @F modified ['df'] which cell @C (earlier in notebook) reads. This violates Sequential Dataflow Consistency.

In [ ]:
# This cell reads len() then modifies - violation!
n = len(df)
print(f"Length: {n}")
df = df.append({'x': 4}, ignore_index=True)  # Changes length

## OK Patterns

These patterns are fine:
- Write first, then read
- Read in one cell, write in a later cell (with proper SDC tracking)

In [ ]:
# OK: Write first, then read
df = pd.DataFrame({'a': [1]})
df['b'] = [2]  # Write first
print(f"Columns: {list(df.columns)}")  # Then read - OK!

In [ ]:
# Reset to warn mode
%structural_tracking warn